### MoMTSim-KAN — Réimplémentation Python/NumPy de MoMTSim
### adaptée aux 5 scénarios de fraude formalisés au Chapitre 3
### du mémoire (ATO, Refund Fraud, Fake Credentials, Split Deposit, Smurfing).

### Dépendances : numpy, pandas

In [2]:
from __future__ import annotations
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from collections import deque
from enum import Enum, auto
from typing import Optional
import bisect
import uuid
import json  # pour fraudScenariosConfig.yaml


In [3]:
RNG = np.random.default_rng(1000)  # graine globale, remplaçable


#### ---------------------------------------------------------------------------
#### 1. UTILITAIRES (équivalents RandomCollection / BoundedArrayDeque)
#### ---------------------------------------------------------------------------


In [4]:

class RandomCollection:
    """Tirage pondéré O(log n) via cumul de poids (équiv. NavigableMap Java)."""
    def __init__(self):
        self._items = []
        self._cum_weights = []
        self._total = 0.0

    def add(self, weight: float, item):
        self._total += weight
        self._cum_weights.append(self._total)
        self._items.append(item)

    def next(self, rng=RNG):
        if not self._items:
            raise ValueError("RandomCollection vide")
        r = rng.uniform(0, self._total)
        idx = bisect.bisect_left(self._cum_weights, r)
        return self._items[min(idx, len(self._items) - 1)]


class BoundedDeque(deque):
    """File bornée à maxlen (équiv. BoundedArrayDeque, 100 par défaut)."""
    def __init__(self, maxlen=100):
        super().__init__(maxlen=maxlen)


#### ---------------------------------------------------------------------------
#### 2. CHARGEMENT DES PARAMÈTRES (les 6 CSV MoMTSim + config fraude YAML)
#### ---------------------------------------------------------------------------


In [6]:
@dataclass
class ClientActionProfile:
    action: str
    profile_id: int
    min_tx: int
    max_tx: int
    mean_amount: float
    std_amount: float
    weight: float


@dataclass
class StepActionProfile:
    step: int
    action: str
    count: int
    total_sum: float
    avg: float
    std: float


class Parameters:
    """Charge les 6 CSV + le YAML de config fraude."""

    def __init__(self, param_dir: str, fraud_config_path: str,
                 seed: int = 1000, n_clients=2000, n_merchants=300,
                 n_banks=20, transfer_limit=5_000_000):
        self.seed = seed
        self.n_clients = n_clients
        self.n_merchants = n_merchants
        self.n_banks = n_banks
        self.transfer_limit = transfer_limit

        self.transaction_types = pd.read_csv(f"{param_dir}/transactionsTypes.csv")["type"].tolist()

        self.client_profiles_df = pd.read_csv(f"{param_dir}/clientsProfiles.csv")
        self.action_profile_pool: dict[str, RandomCollection] = {}
        for action, grp in self.client_profiles_df.groupby("action"):
            rc = RandomCollection()
            for _, row in grp.iterrows():
                cap = ClientActionProfile(
                    action=action, profile_id=row["profile_id"],
                    min_tx=row["min_tx_per_month"], max_tx=row["max_tx_per_month"],
                    mean_amount=row["mean_amount"], std_amount=row["std_amount"],
                    weight=row["weight"])
                rc.add(row["weight"], cap)
            self.action_profile_pool[action] = rc

        agg = pd.read_csv(f"{param_dir}/aggregatedTransactions.csv")
        self.steps_profiles: list[dict[str, StepActionProfile]] = [dict() for _ in range(720)]
        for _, row in agg.iterrows():
            sap = StepActionProfile(row["step"], row["action"], row["count"],
                                     row["sum"], row["avg"], row["std"])
            self.steps_profiles[int(row["step"])][row["action"]] = sap
        self.total_target_count = agg.groupby("step")["count"].sum().to_dict()

        self.initial_balances_df = pd.read_csv(f"{param_dir}/initialBalancesDistribution.csv")
        self._balance_rc = RandomCollection()
        for _, row in self.initial_balances_df.iterrows():
            self._balance_rc.add(row["proportion"], (row["range_min"], row["range_max"]))

        self.overdraft_df = pd.read_csv(f"{param_dir}/overdraftLimits.csv")

        self.max_occ_df = pd.read_csv(f"{param_dir}/maxOccurrencesPerClient.csv")
        self.max_occurrences = dict(zip(self.max_occ_df["action"], self.max_occ_df["max_occurrences"]))

        with open(fraud_config_path, "r", encoding="utf-8") as f:
            self.fraud_config = json.load(f)
            self.fraud_config = yaml.safe_load(f)

    def pick_initial_balance(self, rng=RNG) -> float:
        lo, hi = self._balance_rc.next(rng)
        return float(rng.uniform(lo, hi))

    def get_overdraft_limit(self, mean_amount: float) -> float:
        for _, row in self.overdraft_df.iterrows():
            lo = -np.inf if str(row["mean_amount_min"]) == "-inf" else float(row["mean_amount_min"])
            hi = np.inf if str(row["mean_amount_max"]) == "inf" else float(row["mean_amount_max"])
            if lo <= mean_amount < hi:
                return float(row["overdraft_limit"])
        return 0.0

    def pick_client_profile(self, action: str, rng=RNG) -> Optional[ClientActionProfile]:
        pool = self.action_profile_pool.get(action)
        return pool.next(rng) if pool else None


#### ---------------------------------------------------------------------------
#### 3. TRANSACTION
#### ---------------------------------------------------------------------------


In [8]:

@dataclass
class Transaction:
    step: int
    action: str
    amount: float
    name_orig: str
    old_balance_orig: float
    new_balance_orig: float
    name_dest: str
    old_balance_dest: float
    new_balance_dest: float
    is_fraud: bool = False
    is_flagged_fraud: bool = False
    is_unauthorized_overdraft: bool = False
    is_successful: bool = True
    fraud_scenario: Optional[str] = None  # ATO / REFUND / FAKE_CRED / SPLIT_DEP / SMURFING



#### ---------------------------------------------------------------------------
#### 4. AGENTS
#### ---------------------------------------------------------------------------


In [10]:

class ActorType(Enum):
    BANK = auto()
    CLIENT = auto()
    MERCHANT = auto()
    MULE = auto()
    ATO_FRAUDSTER = auto()
    REFUND_FRAUDSTER = auto()
    FAKE_CRED_FRAUDSTER = auto()
    SPLIT_DEP_FRAUDSTER = auto()
    SMURF_EMITTER = auto()
    SMURF_RECEIVER = auto()


class SuperActor:
    def __init__(self, actor_id: str, actor_type: ActorType, balance: float = 0.0,
                 overdraft_limit: float = 0.0, mean_amount: float = 0.0):
        self.id = actor_id
        self.type = actor_type
        self.balance = balance
        self.overdraft_limit = overdraft_limit
        self.mean_amount = mean_amount
        self.history = BoundedDeque(maxlen=100)  # derniers contacts
        self.tx_count_lifetime: dict[str, int] = {}
        self.high_risk = False

    def can_withdraw(self, amount: float) -> bool:
        return (self.balance - amount) >= self.overdraft_limit

    def deposit(self, amount: float):
        self.balance += amount

    def withdraw(self, amount: float) -> bool:
        if self.can_withdraw(amount):
            self.balance -= amount
            return True
        return False

    def remember(self, other_id: str):
        if other_id not in self.history:
            self.history.append(other_id)


class Merchant(SuperActor):
    def __init__(self, actor_id, high_risk=False):
        super().__init__(actor_id, ActorType.MERCHANT)
        self.high_risk = high_risk
        self.refund_acceptance_proba = float(RNG.uniform(0.3, 0.95))


class Bank(SuperActor):
    def __init__(self, actor_id):
        super().__init__(actor_id, ActorType.BANK)


class Client(SuperActor):
    def __init__(self, actor_id, params: Parameters, rng=RNG):
        balance0 = params.pick_initial_balance(rng)
        # profil par action
        self.profiles: dict[str, ClientActionProfile] = {}
        target_counts = {}
        for action in params.transaction_types:
            prof = params.pick_client_profile(action, rng)
            if prof:
                self.profiles[action] = prof
                target_counts[action] = rng.integers(prof.min_tx, prof.max_tx + 1)
        mean_amount_global = np.mean([p.mean_amount for p in self.profiles.values()]) if self.profiles else 0.0
        overdraft = params.get_overdraft_limit(mean_amount_global)
        super().__init__(actor_id, ActorType.CLIENT, balance=balance0,
                          overdraft_limit=overdraft, mean_amount=mean_amount_global)
        self.target_total_count = sum(target_counts.values())
        self.client_weight = 0.0  # renseigné par MoMTSimState (part du total)
        # état "solde d'équilibre" pour spring model
        self.equilibrium = max(1.0, 40 * mean_amount_global)
        self.in_prob = 0.5
        self.out_prob = 0.5
        self.tx_history_amounts = deque(maxlen=10)  # pour Flag anomalie (µ+2σ glissant)
        self.recent_transfers = deque(maxlen=50)     # pour la règle de fraude native

    # --- 3.3 Spring Model ---
    def spring_probabilities(self):
        k = 1.0 / self.equilibrium
        spring_force = k * (self.equilibrium - self.balance)
        correction_strength = 1.0
        new_prob_in = 0.5 * (1 + correction_strength * spring_force + (self.in_prob - self.out_prob))
        new_prob_in = float(np.clip(new_prob_in, 0.0, 1.0))
        return new_prob_in, 1.0 - new_prob_in

    # --- 3.1 nombre de transactions à ce step (binomiale) ---
    def draw_tx_count(self, step_target_count: int, rng=RNG) -> int:
        if self.target_total_count <= 0 or step_target_count <= 0:
            return 0
        n = int(rng.binomial(step_target_count, min(1.0, self.client_weight)))
        return n

    # --- 3.2 montant (loi normale, profil client + profil step) ---
    def draw_amount(self, action: str, step_profile: Optional[StepActionProfile], rng=RNG) -> float:
        prof = self.profiles.get(action)
        if prof is None:
            return 0.0
        if step_profile is not None:
            mu = (prof.mean_amount + step_profile.avg) / 2
            sigma = np.sqrt((prof.std_amount**2 + step_profile.std**2)) / 2
        else:
            mu, sigma = prof.mean_amount, prof.std_amount
        amount = rng.normal(mu, max(sigma, 1e-6))
        while amount <= 0:
            amount = rng.normal(mu, max(sigma, 1e-6))
        return float(amount)

    # --- 3.4 Stickiness ---
    def pick_counterparty(self, candidate_pool: list[str], rng=RNG) -> str:
        if self.history and rng.uniform() < 0.90:
            return rng.choice(list(self.history))
        target = rng.choice(candidate_pool)
        if target not in self.history and rng.uniform() < 0.90:
            self.remember(target)
        return target

    # --- 3.6 règle de détection native ---
    def check_native_fraud_flag(self, amount: float, transfer_limit: float) -> bool:
        self.recent_transfers.append(amount)
        if len(self.recent_transfers) >= 3:
            balance_max = max(self.recent_transfers) if self.recent_transfers else self.balance
            if (balance_max - self.balance - amount) > transfer_limit * 2.5:
                return True
        return False

    def update_amount_history(self, amount: float):
        self.tx_history_amounts.append(amount)

    def anomaly_flag(self, amount: float) -> bool:
        if len(self.tx_history_amounts) < 2:
            return False
        mu = np.mean(self.tx_history_amounts)
        sigma = np.std(self.tx_history_amounts)
        return amount > (mu + 2 * sigma)


class Mule(Client):
    """Compte fantoche : step() n'agit pas seul, activé par un fraudeur."""
    def __init__(self, actor_id, params: Parameters, rng=RNG):
        super().__init__(actor_id, params, rng)
        self.type = ActorType.MULE
        self.controller_id: Optional[str] = None

    def fraudulent_cash_out(self, amount: float) -> bool:
        return self.withdraw(amount)




#### ---------------------------------------------------------------------------
#### 5. AGENTS FRAUDEURS — formules EXACTES de la section 3.2 du mémoire
#### ---------------------------------------------------------------------------


In [12]:

class BaseFraudster:
    scenario_name = "BASE"

    def __init__(self, cfg: dict, params: Parameters):
        self.cfg = cfg
        self.params = params

    def make_tx(self, step, action, amount, orig: SuperActor, dest: SuperActor,
                fraud=True, flagged=False) -> Transaction:
        old_o, old_d = orig.balance, dest.balance
        success = orig.withdraw(amount)
        if success:
            dest.deposit(amount)
        return Transaction(
            step=step, action=action, amount=amount,
            name_orig=orig.id, old_balance_orig=old_o, new_balance_orig=orig.balance,
            name_dest=dest.id, old_balance_dest=old_d, new_balance_dest=dest.balance,
            is_fraud=fraud, is_flagged_fraud=flagged, is_successful=success,
            fraud_scenario=self.scenario_name)


class ATOFraudster(BaseFraudster):
    """3.2.1 — Account Takeover : retraits massifs haute vélocité."""
    scenario_name = "ATO"

    def __init__(self, cfg, params):
        super().__init__(cfg["ato"], params)

    def execute(self, step: int, victim: Client, mule_pool: list[Mule], rng=RNG) -> list[Transaction]:
        c = self.cfg
        if victim.balance < c["B_min"]:
            return []
        B0 = victim.balance
        n = int(rng.integers(c["n_min"], c["n_max"] + 1))
        mules = list(rng.choice(mule_pool, size=min(n, len(mule_pool)), replace=False))
        # fragments non vus auparavant (nouveauté du destinataire garantie par choix pool mules)
        remaining = B0
        txs = []
        t = step
        for mule in mules:
            if remaining <= 0:
                break
            frac = rng.uniform(c["frag_min"], c["frag_max"])
            amount = min(remaining, frac * B0)
            if amount <= 0:
                continue
            dt = rng.exponential(1.0 / c["lambda_ato"])  # en fraction de step (heures)
            tx = self.make_tx(t, "TRANSFER", amount, victim, mule, fraud=True)
            txs.append(tx)
            remaining -= amount
            t = step  # toutes les tx restent horodatées dans la fenêtre <10 min = même step
        return txs


class RefundFraudster(BaseFraudster):
    """3.2.2 — Refund Fraud : boucles paiement/remboursement."""
    scenario_name = "REFUND"

    def __init__(self, cfg, params):
        super().__init__(cfg["refund"], params)
        self.vuln_registry: dict[str, list[str]] = {}  # fraudster_id -> [merchant_ids]
        self.cycle_counts: dict[str, int] = {}

    def select_vulnerable_merchants(self, fraudster_id: str, merchants: list[Merchant]):
        thresh = self.cfg["p_refund_threshold"]
        vuln = [m.id for m in merchants if m.refund_acceptance_proba > thresh]
        self.vuln_registry[fraudster_id] = vuln

    def execute(self, step: int, fraudster_id: str, orig: SuperActor,
                merchants: dict[str, Merchant], rng=RNG) -> list[Transaction]:
        c = self.cfg
        vuln = self.vuln_registry.get(fraudster_id, [])
        if not vuln:
            return []
        m_id = rng.choice(vuln)
        merchant = merchants[m_id]
        amount = float(rng.normal(3325, 800))  # ancré sur moyenne paiement marchand du mémoire
        amount = max(amount, 100)
        pay_tx = self.make_tx(step, "PAYMENT", amount, orig, merchant, fraud=True)
        refund_tx = self.make_tx(step, "REFUND", amount, merchant, orig, fraud=True)
        self.cycle_counts[fraudster_id] = self.cycle_counts.get(fraudster_id, 0) + 1
        if self.cycle_counts[fraudster_id] >= c["k_max"]:
            vuln.remove(m_id)
            self.cycle_counts[fraudster_id] = 0
        return [pay_tx, refund_tx]


class FakeCredentialsFraudster(BaseFraudster):
    """3.2.3 — Fake Credentials : dormance puis exfiltration."""
    scenario_name = "FAKE_CRED"

    def __init__(self, cfg, params):
        super().__init__(cfg["fake_credentials"], params)

    def execute_dormant_tx(self, step: int, actor: Client, merchant: Merchant, rng=RNG) -> Transaction:
        c = self.cfg
        amount = float(rng.uniform(500, c["m_leg_max"]))
        return self.make_tx(step, "PAYMENT", amount, actor, merchant, fraud=False)

    def execute_exfiltration(self, step: int, actor: Client, dest: SuperActor,
                              plafond: float, rng=RNG) -> Transaction:
        c = self.cfg
        amount = float(rng.uniform(c["m_exp_ratio_min"] * plafond, plafond))
        return self.make_tx(step, "TRANSFER", amount, actor, dest, fraud=True, flagged=True)


class SplitDepositFraudster(BaseFraudster):
    """3.2.4 — Split Deposit : arbitrage de commission par agent."""
    scenario_name = "SPLIT_DEP"

    def __init__(self, cfg, params):
        super().__init__(cfg["split_deposit"], params)
        self.tariff_grid: list[tuple[float, float]] = [
            (float(t["threshold"]), float(t["commission"])) for t in self.cfg["tariff_grid"]
        ]

    def _commission(self, amount: float) -> float:
        c = 0.0
        for threshold, comm in self.tariff_grid:
            if amount >= threshold:
                c = comm
        return c

    def optimal_fragmentation(self, total: float, rng=RNG) -> list[float]:
        best_k, best_gain = 1, self._commission(total)
        for k in range(2, 6):
            frag = total / k
            gain_k = k * self._commission(frag)
            if gain_k > best_gain:
                best_gain, best_k = gain_k, k
        eps = rng.uniform(0, self.cfg["epsilon_max"])
        base = total / best_k
        fragments = [base + rng.uniform(-eps, eps) for _ in range(best_k - 1)]
        fragments.append(total - sum(fragments))
        return [max(f, 1.0) for f in fragments]

    def execute(self, step: int, client: Client, agent: SuperActor,
                total_deposit: float, rng=RNG) -> list[Transaction]:
        fragments = self.optimal_fragmentation(total_deposit, rng)
        return [self.make_tx(step, "CASH_IN", f, agent, client, fraud=True) for f in fragments]


class SmurfingNetwork(BaseFraudster):
    """3.2.5 — Smurfing : réseau f1 -> mules -> f2, critères de Zhdanova et al."""
    scenario_name = "SMURFING"

    def __init__(self, cfg, params, emitter: SuperActor, receiver: SuperActor,
                 mules: list[Mule], n_conscious_ratio=0.6):
        super().__init__(cfg["smurfing"], params)
        self.emitter = emitter
        self.receiver = receiver
        self.mules = mules
        n_conscious = int(len(mules) * n_conscious_ratio)
        self.conscious_mules = set(m.id for m in mules[:n_conscious])

    def run_laundering_operation(self, step: int, total_X: float, rng=RNG) -> list[Transaction]:
        c = self.cfg
        s_seuil = c["S_seuil"]
        k = int(rng.integers(2, len(self.mules) + 1))
        chosen = list(rng.choice(self.mules, size=min(k, len(self.mules)), replace=False))
        fractions = rng.uniform(0.70 * s_seuil, 0.99 * s_seuil, size=len(chosen))
        fractions = fractions / fractions.sum() * min(total_X, len(chosen) * 0.99 * s_seuil)

        txs = []
        for mule, x_i in zip(chosen, fractions):
            tx_in = self.make_tx(step, "TRANSFER", float(x_i), self.emitter, mule, fraud=True)
            txs.append(tx_in)
            delta_i = rng.uniform(c["delta_min"], c["delta_max"])
            amount_out = float(x_i) * (1 - delta_i)
            tx_out = self.make_tx(step, "TRANSFER", amount_out, mule, self.receiver, fraud=True)
            txs.append(tx_out)
        return txs

#### Calibration par minimisation du SSE (section 3.1.3)
#### Remplace le contrôleur proportionnel par le vrai protocole : minimisation
#### de la somme des erreurs quadratiques entre distribution cible et distribution
#### simulée, par scénario et par fenêtre temporelle.

In [13]:
class SSEFraudCalibrator:
    SCENARIOS = ["ato", "refund", "fake_credentials", "split_deposit", "smurfing"]
    SCENARIO_LABELS = {"ato": "ATO", "refund": "REFUND", "fake_credentials": "FAKE_CRED",
                        "split_deposit": "SPLIT_DEP", "smurfing": "SMURFING"}

    def __init__(self, param_dir: str, fraud_config_path: str, seed: int = 1000,
                 n_clients=500, n_merchants=100, n_banks=10, n_mules=30,
                 target_mid=0.23, n_steps=720, n_bins=30, n_seeds_per_eval=3):
        """
        n_bins=30 : fenêtres journalières (24 steps chacune) sur les 720 steps,
        cohérent avec la granularité utilisée pour Dr/Ds en 3.1.3.
        n_seeds_per_eval : moyenne sur plusieurs graines pour lisser le bruit
        stochastique de la fonction objectif (nécessaire pour la stabilité
        de l'optimiseur sur une fonction non déterministe).
        """
        self.param_dir = param_dir
        self.fraud_config_path = fraud_config_path
        self.seed = seed
        self.n_clients = n_clients
        self.n_merchants = n_merchants
        self.n_banks = n_banks
        self.n_mules = n_mules
        self.target_mid = target_mid
        self.n_steps = n_steps
        self.n_bins = n_bins
        self.bin_size = n_steps // n_bins
        self.n_seeds_per_eval = n_seeds_per_eval

        # target Dr(c,t) : construite une seule fois, à partir de aggregatedTransactions.csv
        self._Dr = None
        self._legit_target_per_bin = None

    # ------------------------------------------------------------------
    def _build_target_distribution(self, sim_probe: MoMTSimState) -> np.ndarray:
        """Dr(c,t) : distribution cible de fraude par scénario (5) x bin (n_bins).
        Basée sur le volume légitime attendu (aggregatedTransactions.csv, déjà
        chargé dans Parameters.total_target_count) et le taux cible global,
        répartie équitablement sur les 5 scénarios (section 3.1.1).
        Hypothèse documentée : en l'absence de données réelles de fraude
        (section 1.7.1), la forme temporelle est supposée proportionnelle
        au volume légitime attendu par bin plutôt qu'uniforme.
        """
        legit_per_step = np.array([sim_probe.params.total_target_count.get(s, 0)
                                    for s in range(self.n_steps)])
        legit_per_bin = legit_per_step.reshape(self.n_bins, self.bin_size).sum(axis=1)
        self._legit_target_per_bin = legit_per_bin

        fraud_ratio = self.target_mid / (1 - self.target_mid)
        fraud_total_per_bin = legit_per_bin * fraud_ratio

        # équirépartition stricte entre les 5 scénarios (section 3.1.1)
        Dr = np.tile(fraud_total_per_bin / len(self.SCENARIOS), (len(self.SCENARIOS), 1))
        return Dr  # shape (5, n_bins)

    # ------------------------------------------------------------------
    def _run_trial_binned(self, theta: np.ndarray, seed_offset: int) -> np.ndarray:
        """Exécute un run et retourne Ds(c,t;θ) : compte de tx frauduleuses
        par scénario (5) x bin (n_bins)."""
        probas = {
            "ato": theta[0], "refund": theta[1], "fake_credentials": theta[2],
            "split_deposit": theta[3], "smurfing_freq_mult": theta[4],
        }

        sim = MoMTSimState(
            param_dir=self.param_dir, fraud_config_path=self.fraud_config_path,
            seed=self.seed + seed_offset, n_clients=self.n_clients,
            n_merchants=self.n_merchants, n_banks=self.n_banks, n_mules=self.n_mules)
        sim.fraud_probas = probas  # utilisé si momtsim.py a été patché (voir partie 2)

        df = sim.run(n_steps=self.n_steps, verbose=False)
        Ds = np.zeros((len(self.SCENARIOS), self.n_bins))
        if len(df) == 0:
            return Ds

        fraud_df = df[df["isFraud"]].copy()
        if len(fraud_df) == 0:
            return Ds

        fraud_df["bin"] = (fraud_df["step"] // self.bin_size).clip(upper=self.n_bins - 1)
        for i, key in enumerate(self.SCENARIOS):
            label = self.SCENARIO_LABELS[key]
            counts = fraud_df[fraud_df["fraudScenario"] == label].groupby("bin").size()
            for b, c in counts.items():
                Ds[i, int(b)] = c
        return Ds

    # ------------------------------------------------------------------
    def _objective(self, theta: np.ndarray) -> float:
        theta = np.clip(theta, 1e-4, None)
        Ds_list = [self._run_trial_binned(theta, seed_offset=k)
                   for k in range(self.n_seeds_per_eval)]
        Ds_mean = np.mean(Ds_list, axis=0)
        sse = float(np.sum((self._Dr - Ds_mean) ** 2))
        print(f"θ={np.round(theta,4)} -> SSE={sse:,.1f}", flush=True)
        return sse

    # ------------------------------------------------------------------
    def calibrate(self, x0=None, maxiter=25, lr=0.05, spsa_c=0.02) -> dict:
        """
        Optimisation SPSA (Simultaneous Perturbation Stochastic Approximation)
        en PyTorch — adaptée car la simulation est bruitée et non différentiable.
        À chaque itération : deux évaluations (theta+perturbation, theta-perturbation)
        suffisent pour estimer un gradient approché, quel que soit le nombre de
        paramètres (contrairement aux différences finies classiques qui coûteraient
        2*dim évaluations).
        """
        sim_probe = MoMTSimState(
            param_dir=self.param_dir, fraud_config_path=self.fraud_config_path,
            seed=self.seed, n_clients=self.n_clients, n_merchants=self.n_merchants,
            n_banks=self.n_banks, n_mules=self.n_mules)
        self._Dr = self._build_target_distribution(sim_probe)

        bounds_lo = torch.tensor([1e-4, 1e-4, 1e-4, 1e-4, 0.1])
        bounds_hi = torch.tensor([0.5, 0.5, 0.3, 0.5, 10.0])

        if x0 is None:
            theta = torch.tensor([0.02, 0.02, 0.005, 0.03, 1.0], dtype=torch.float32)
        else:
            theta = torch.tensor(x0, dtype=torch.float32)

        best_theta, best_sse = theta.clone(), float("inf")
        history = []

        for it in range(maxiter):
            # perturbation aléatoire de Bernoulli (+1/-1), condition SPSA classique
            delta = torch.tensor(np.random.choice([-1.0, 1.0], size=5), dtype=torch.float32)

            theta_plus = torch.clamp(theta + spsa_c * delta, bounds_lo, bounds_hi)
            theta_minus = torch.clamp(theta - spsa_c * delta, bounds_lo, bounds_hi)

            sse_plus = self._objective(theta_plus.numpy())
            sse_minus = self._objective(theta_minus.numpy())

            # estimation du gradient SPSA
            grad_hat = (sse_plus - sse_minus) / (2 * spsa_c * delta)
            grad_hat = torch.tensor(grad_hat, dtype=torch.float32)

            theta = theta - lr * grad_hat
            theta = torch.clamp(theta, bounds_lo, bounds_hi)

            sse_current = self._objective(theta.numpy())
            history.append({"iter": it, "sse": sse_current, "theta": theta.tolist()})
            print(f"[iter {it}] θ={theta.tolist()} SSE={sse_current:,.1f}")

            if sse_current < best_sse:
                best_sse, best_theta = sse_current, theta.clone()

        theta_star = best_theta.numpy()
        probas_final = {
            "ato": float(theta_star[0]), "refund": float(theta_star[1]),
            "fake_credentials": float(theta_star[2]), "split_deposit": float(theta_star[3]),
            "smurfing_freq_mult": float(theta_star[4]),
        }
        return {"probas": probas_final, "sse_final": float(best_sse),
                "converged": best_sse < np.inf, "history": history}

#### Ingénierie des caractéristiques transactionnelles
#### (section 3.2.6 du mémoire), calculées à partir de rawLog.csv.


In [17]:
class FeatureEngineer:
    """
    Calcule, pour chaque transaction du rawLog, l'ensemble des features
    dérivées formalisées en section 3.2.6 :
    ΔBorig, ΔBdest, r1, r2, Flag_anomalie, δ_commission, Var_agent,
    ρ_rupture, ρ_refund, V1h, Flag_nuit, ρ_nouveau.
    """

    def __init__(self, df: pd.DataFrame, eps: float = 1e-6):
        self.df = df.copy().sort_values(["step"]).reset_index(drop=True)
        self.eps = eps

    # ------------------------------------------------------------------
    def compute_all(self) -> pd.DataFrame:
        df = self.df
        df = self._delta_balances(df)
        df = self._ratios_r1_r2(df)
        df = self._flag_anomalie(df)
        df = self._flag_nuit(df)
        df = self._velocity_1h(df)
        df = self._delta_commission_smurfing(df)
        df = self._var_agent_split_deposit(df)
        df = self._rho_rupture_fake_cred(df)
        df = self._rho_refund(df)
        df = self._rho_nouveau(df)
        self.df = df
        return df

    # ------------------------------------------------------------------
    # ΔBorig, ΔBdest — 3.2.6, éq. 3.8 / 3.9
    # ------------------------------------------------------------------
    def _delta_balances(self, df: pd.DataFrame) -> pd.DataFrame:
        df["delta_B_orig"] = df["oldBalanceOrig"] - df["newBalanceOrig"]
        df["delta_B_dest"] = df["newBalanceDest"] - df["oldBalanceDest"]
        return df

    # ------------------------------------------------------------------
    # r1, r2 — éq. 3.10 / 3.11 (feature clé ATO)
    # ------------------------------------------------------------------
    def _ratios_r1_r2(self, df: pd.DataFrame) -> pd.DataFrame:
        df["r1"] = df["amount"] / (df["oldBalanceOrig"] + self.eps)
        df["r2"] = df["amount"] / (df["newBalanceOrig"] + self.eps)
        df["r1_r2_product"] = df["r1"] * df["r2"]  # exploitable par un nœud MultKAN
        return df

    # ------------------------------------------------------------------
    # Flag_anomalie — éq. 3.12, fenêtre glissante des 10 dernières opérations
    # du même compte ET du même type de transaction
    # ------------------------------------------------------------------
    def _flag_anomalie(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.sort_values(["nameOrig", "action", "step"]).reset_index(drop=True)
        grp = df.groupby(["nameOrig", "action"])["amount"]
        # rolling sur les 10 dernières tx AVANT la courante (shift pour éviter la fuite)
        rolling_mean = grp.transform(lambda s: s.shift(1).rolling(10, min_periods=2).mean())
        rolling_std = grp.transform(lambda s: s.shift(1).rolling(10, min_periods=2).std())
        threshold = rolling_mean + 2 * rolling_std
        df["flag_anomalie"] = (df["amount"] > threshold).fillna(False)
        return df.sort_values("step").reset_index(drop=True)

    # ------------------------------------------------------------------
    # Flag_nuit — éq. 3.18
    # ------------------------------------------------------------------
    def _flag_nuit(self, df: pd.DataFrame) -> pd.DataFrame:
        hour_of_day = df["step"] % 24
        df["flag_nuit"] = ((hour_of_day >= 22) | (hour_of_day < 6))
        return df

    # ------------------------------------------------------------------
    # V1h — éq. 3.17, nombre de tx du compte émetteur dans la fenêtre glissante 1h
    # (ici : le step précédent inclus, puisque 1 step = 1h dans ce simulateur)
    # ------------------------------------------------------------------
    def _velocity_1h(self, df: pd.DataFrame) -> pd.DataFrame:
        counts_per_step = df.groupby(["nameOrig", "step"]).size().rename("v1h_raw")
        df = df.merge(counts_per_step, on=["nameOrig", "step"], how="left")
        df["v1h"] = df["v1h_raw"]
        df = df.drop(columns=["v1h_raw"])
        return df

    # ------------------------------------------------------------------
    # δ_commission — éq. 3.13, feature Smurfing
    # Appariement transaction TRANSFER entrante la plus récente / sortante
    # la plus proche pour un même compte (candidat mule)
    # ------------------------------------------------------------------
    def _delta_commission_smurfing(self, df: pd.DataFrame) -> pd.DataFrame:
        df["delta_commission"] = np.nan
        df["delta_commission_ratio"] = np.nan
        df["is_mule_candidate"] = False

        transfers = df[df["action"] == "TRANSFER"].copy()
        if transfers.empty:
            return df

        in_tx = transfers[["nameDest", "step", "amount"]].rename(
            columns={"nameDest": "account", "amount": "amount_in", "step": "step_in"})
        out_tx = transfers[["nameOrig", "step", "amount"]].rename(
            columns={"nameOrig": "account", "amount": "amount_out", "step": "step_out"})

        in_tx = in_tx.sort_values(["account", "step_in"])
        out_tx = out_tx.sort_values(["account", "step_out"])

        # appariement asof : pour chaque sortie, la dernière entrée précédente sur le même compte
        merged = pd.merge_asof(
            out_tx.sort_values("step_out"), in_tx.sort_values("step_in"),
            left_on="step_out", right_on="step_in", by="account", direction="backward")
        merged["delta"] = merged["amount_in"] - merged["amount_out"]
        merged["delta_ratio"] = merged["delta"] / (merged["amount_in"] + self.eps)

        # Critère 1 de Zhdanova et al. : 0 < δ ≤ 10% du montant reçu
        merged["is_mule_candidate"] = (merged["delta_ratio"] > 0) & (merged["delta_ratio"] <= 0.10)

        agg = merged.groupby("account").agg(
            delta_commission_ratio=("delta_ratio", "last"),
            is_mule_candidate=("is_mule_candidate", "any")).reset_index()

        df = df.merge(
            agg.rename(columns={"account": "nameOrig"}),
            on="nameOrig", how="left", suffixes=("", "_mule"))
        df["delta_commission_ratio"] = df["delta_commission_ratio_mule"].combine_first(
            df["delta_commission_ratio"])
        df["is_mule_candidate"] = df["is_mule_candidate_mule"].fillna(False)
        df = df.drop(columns=[c for c in df.columns if c.endswith("_mule")])
        return df

    # ------------------------------------------------------------------
    # Var_agent — éq. 3.14, Split Deposit
    # Regroupement des CASH_IN au même step, même agent (orig), même client (dest)
    # -> équivalent de la fenêtre 60-120s puisque le simulateur les génère
    # simultanément au même step (limitation de granularité horaire assumée)
    # ------------------------------------------------------------------
    def _var_agent_split_deposit(self, df: pd.DataFrame) -> pd.DataFrame:
        df["var_agent_split"] = np.nan
        df["k_fragments"] = 0

        cash_in = df[df["action"] == "CASH_IN"].copy()
        if cash_in.empty:
            return df

        grp = cash_in.groupby(["nameOrig", "nameDest", "step"])["amount"]
        stats = grp.agg(var_agent_split="var", k_fragments="count").reset_index()
        stats["var_agent_split"] = stats["var_agent_split"].fillna(0.0)

        df = df.merge(stats, on=["nameOrig", "nameDest", "step"], how="left",
                       suffixes=("", "_new"))
        df["var_agent_split"] = df["var_agent_split_new"].combine_first(df["var_agent_split"])
        df["k_fragments"] = df["k_fragments_new"].fillna(0).astype(int)
        df = df.drop(columns=[c for c in df.columns if c.endswith("_new")])
        return df

    
    # ------------------------------------------------------------------
    # Helper vectorisé : somme/compte sur fenêtre glissante par borne de step,
    # via cumsum + searchsorted (O(n log n), aucune boucle Python par ligne)
    # ------------------------------------------------------------------
    @staticmethod
    def _windowed_sum_by_group(steps: np.ndarray, values: np.ndarray,
                                lo_bounds: np.ndarray, hi_bounds: np.ndarray) -> np.ndarray:
        """
        steps, values : triés par step, déjà filtrés sur le groupe (ex: un compte).
        lo_bounds, hi_bounds : pour chaque requête i, la fenêtre [lo_bounds[i], hi_bounds[i]).
        Retourne la somme de `values` dont step ∈ [lo, hi) pour chaque requête.
        """
        cumsum = np.concatenate([[0.0], np.cumsum(values)])
        idx_lo = np.searchsorted(steps, lo_bounds, side="left")
        idx_hi = np.searchsorted(steps, hi_bounds, side="left")
        return cumsum[idx_hi] - cumsum[idx_lo]
    
    
    # ------------------------------------------------------------------
    # ρ_rupture — éq. 3.15, Fake Credentials
    # Moyenne historique sur 30j glissants AVANT la transaction courante
    # ------------------------------------------------------------------
    def _rho_rupture_fake_cred(self, df: pd.DataFrame) -> pd.DataFrame:
        window_steps = 30 * 24
        df = df.sort_values(["nameOrig", "step"]).reset_index(drop=True)

        mean_hist = np.full(len(df), np.nan)
        for _, idx in df.groupby("nameOrig").indices.items():
            idx = np.sort(idx)
            steps_g = df["step"].values[idx]
            amounts_g = df["amount"].values[idx]

            lo = steps_g - window_steps
            hi = steps_g  # exclusif : strictement avant la transaction courante

            sums = self._windowed_sum_by_group(steps_g, amounts_g, lo, hi)
            counts = self._windowed_sum_by_group(steps_g, np.ones_like(amounts_g), lo, hi)
            means = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
            mean_hist[idx] = means

        df["mean_historique_30j"] = mean_hist
        df["rho_rupture"] = df["amount"] / (df["mean_historique_30j"].fillna(0) + self.eps)
        return df

    # ------------------------------------------------------------------
    # ρ_refund — éq. 3.16, fenêtre glissante 30j
    # ------------------------------------------------------------------
    def _rho_refund(self, df: pd.DataFrame) -> pd.DataFrame:
        window_steps = 30 * 24
        df = df.sort_values(["nameOrig", "step"]).reset_index(drop=True)

        is_refund = (df["action"].values == "REFUND").astype(float)
        is_payment = (df["action"].values == "PAYMENT").astype(float)
        rho = np.zeros(len(df))

        for _, idx in df.groupby("nameOrig").indices.items():
            idx = np.sort(idx)
            steps_g = df["step"].values[idx]
            refund_g = is_refund[idx]
            payment_g = is_payment[idx]

            lo = steps_g - window_steps
            hi = steps_g + 1  # inclusif : la transaction courante compte

            n_refund = self._windowed_sum_by_group(steps_g, refund_g, lo, hi)
            n_payment = self._windowed_sum_by_group(steps_g, payment_g, lo, hi)
            rho[idx] = n_refund / (n_payment + self.eps)

        df["rho_refund"] = rho
        return df

    # ------------------------------------------------------------------
    # ρ_nouveau — éq. 3.19, ratio de destinataires inconnus sur 30/90j
    # ------------------------------------------------------------------
    def _rho_nouveau(self, df: pd.DataFrame) -> pd.DataFrame:
        window_hist = 90 * 24
        window_recent = 30 * 24
        df = df.sort_values(["nameOrig", "step"]).reset_index(drop=True)

        # étape 1 (vectorisée) : dernier contact avec ce même (nameOrig, nameDest)
        # avant la transaction courante -> is_new_dest = pas de contact dans les 90j précédents
        last_contact_step = df.groupby(["nameOrig", "nameDest"])["step"].shift(1)
        gap = df["step"] - last_contact_step
        is_new_dest = (last_contact_step.isna() | (gap > window_hist)).astype(float).values

        # étape 2 (vectorisée) : moyenne glissante de is_new_dest sur la fenêtre 30j
        # par compte émetteur, via searchsorted/cumsum
        rho = np.zeros(len(df))
        for _, idx in df.groupby("nameOrig").indices.items():
            idx = np.sort(idx)
            steps_g = df["step"].values[idx]
            novelty_g = is_new_dest[idx]

            lo = steps_g - window_recent
            hi = steps_g + 1  # inclusif

            sums = self._windowed_sum_by_group(steps_g, novelty_g, lo, hi)
            counts = self._windowed_sum_by_group(steps_g, np.ones_like(novelty_g), lo, hi)
            rho[idx] = sums / np.maximum(counts, 1)

        df["rho_nouveau"] = rho
        return df

#### Orchestrateur — boucle de simulation 720 steps (MoMTSimState équivalent)


In [15]:
class MoMTSimState:
    def __init__(self, param_dir: str, fraud_config_path: str,
                 seed: int = 1000, n_clients=2000, n_merchants=300,
                 n_banks=20, n_mules=60, calibrated_probas_path: Optional[str] = None):
        self.rng = np.random.default_rng(seed)
        self.params = Parameters(param_dir, fraud_config_path, seed=seed,
                                  n_clients=n_clients, n_merchants=n_merchants,
                                  n_banks=n_banks)
        self.n_mules = n_mules

        # probas de déclenchement calibrées (section 3.1.3) — fallback si absent
        default_probas = {"ato": 0.02, "refund": 0.02, "fake_credentials": 0.005,
                           "split_deposit": 0.03, "smurfing_freq_mult": 1.0}
        if calibrated_probas_path is not None:
            with open(calibrated_probas_path, "r", encoding="utf-8") as f:
                self.fraud_probas = json.load(f)
        else:
            self.fraud_probas = default_probas

        # --- Population ---
        self.banks = [Bank(f"B{i}") for i in range(n_banks)]
        self.merchants = {f"M{i}": Merchant(f"M{i}", high_risk=(self.rng.uniform() < 0.90))
                           for i in range(n_merchants)}
        self.clients: dict[str, Client] = {}
        for i in range(n_clients):
            cid = f"C{i}"
            self.clients[cid] = Client(cid, self.params, self.rng)
        total_target = sum(c.target_total_count for c in self.clients.values())
        for c in self.clients.values():
            c.client_weight = c.target_total_count / total_target if total_target > 0 else 0.0

        self.mules = [Mule(f"MU{i}", self.params, self.rng) for i in range(n_mules)]

        # --- Fraudeurs (agents de contrôle, pas des acteurs financiers en soi) ---
        cfg = self.params.fraud_config
        self.ato = ATOFraudster(cfg, self.params)
        self.refund = RefundFraudster(cfg, self.params)
        self.refund.select_vulnerable_merchants("global_refund", list(self.merchants.values()))
        self.fake_cred = FakeCredentialsFraudster(cfg, self.params)
        self.split_dep = SplitDepositFraudster(cfg, self.params)

        # réseaux de smurfing pré-instanciés (quelques réseaux actifs en parallèle)
        self.smurf_networks: list[SmurfingNetwork] = []
        n_networks = 5
        client_ids = list(self.clients.keys())
        for _ in range(n_networks):
            emitter = self.clients[self.rng.choice(client_ids)]
            receiver = self.clients[self.rng.choice(client_ids)]
            k = int(self.rng.integers(cfg["smurfing"]["n_mules_min"], cfg["smurfing"]["n_mules_max"] + 1))
            net_mules = list(self.rng.choice(self.mules, size=min(k, len(self.mules)), replace=False))
            self.smurf_networks.append(SmurfingNetwork(cfg, self.params, emitter, receiver,
                                                         net_mules, cfg["smurfing"]["pct_conscious"]))

        # état pour scénarios porteurs de mémoire (dormance, cycles...)
        self._fake_cred_agents: dict[str, dict] = {}   # client_id -> {"dormant_until": step, "activated": bool}
        self._split_dep_agents: dict[str, SuperActor] = {ag.id: ag for ag in self.merchants.values()
                                                            if ag.high_risk}
        self._smurf_next_op: dict[int, int] = {i: int(self.rng.integers(0, 30 * 24))
                                                 for i in range(len(self.smurf_networks))}

        self.all_transactions: list[Transaction] = []
        self.transfer_limit = self.params.transfer_limit

    # ------------------------------------------------------------------
    def _step_target_count(self, step: int) -> int:
        return int(self.params.total_target_count.get(step, 0))

    def _step_action_profile(self, step: int, action: str) -> Optional[StepActionProfile]:
        return self.params.steps_profiles[step].get(action) if step < 720 else None

    # ------------------------------------------------------------------
    def _run_client_legit_step(self, step: int, client: Client):
        n_tx = client.draw_tx_count(self._step_target_count(step), self.rng)
        for _ in range(n_tx):
            p_in, p_out = client.spring_probabilities()
            actions_available = list(client.profiles.keys())
            if not actions_available:
                continue
            action = self.rng.choice(actions_available)
            step_prof = self._step_action_profile(step, action)
            amount = client.draw_amount(action, step_prof, self.rng)
            if amount <= 0:
                continue

            counterpart_pool = list(self.merchants.keys()) if action in ("PAYMENT", "CASH_IN") \
                else list(self.clients.keys())
            dest_id = client.pick_counterparty(counterpart_pool, self.rng)
            dest = self.merchants.get(dest_id) or self.clients.get(dest_id)
            if dest is None or dest.id == client.id:
                continue

            if not client.can_withdraw(amount):
                self.all_transactions.append(Transaction(
                    step=step, action=action, amount=amount,
                    name_orig=client.id, old_balance_orig=client.balance,
                    new_balance_orig=client.balance, name_dest=dest.id,
                    old_balance_dest=dest.balance, new_balance_dest=dest.balance,
                    is_successful=False, is_unauthorized_overdraft=True))
                continue

            old_o, old_d = client.balance, dest.balance
            client.withdraw(amount)
            dest.deposit(amount)
            client.update_amount_history(amount)
            flagged = False
            if action == "TRANSFER":
                flagged = client.check_native_fraud_flag(amount, self.transfer_limit)

            self.all_transactions.append(Transaction(
                step=step, action=action, amount=amount,
                name_orig=client.id, old_balance_orig=old_o, new_balance_orig=client.balance,
                name_dest=dest.id, old_balance_dest=old_d, new_balance_dest=dest.balance,
                is_fraud=False, is_flagged_fraud=flagged, is_successful=True))

    # ------------------------------------------------------------------
    def _inject_fraud(self, step: int):
        cfg = self.params.fraud_config
        client_ids = list(self.clients.keys())

        # --- ATO ---
        if self.rng.uniform() < self.fraud_probas["ato"]:
            victim = self.clients[self.rng.choice(client_ids)]
            txs = self.ato.execute(step, victim, self.mules, self.rng)
            self.all_transactions.extend(txs)

        # --- Refund Fraud ---
        if self.rng.uniform() < self.fraud_probas["refund"]:
            fraudster = self.clients[self.rng.choice(client_ids)]
            txs = self.refund.execute(step, "global_refund", fraudster, self.merchants, self.rng)
            self.all_transactions.extend(txs)

        # --- Fake Credentials ---
        if self.rng.uniform() < self.fraud_probas["fake_credentials"] and len(self._fake_cred_agents) < 200:
            cid = self.rng.choice(client_ids)
            if cid not in self._fake_cred_agents:
                dormance_h = int(self.rng.integers(cfg["fake_credentials"]["dormance_min_days"] * 24,
                                                     cfg["fake_credentials"]["dormance_max_days"] * 24))
                self._fake_cred_agents[cid] = {
                    "dormant_until": step + dormance_h,
                    "n_leg_done": 0,
                    "n_leg_target": int(self.rng.integers(cfg["fake_credentials"]["n_leg_min"],
                                                            cfg["fake_credentials"]["n_leg_max"] + 1)),
                    "activated": False,
                }
        for cid, state in list(self._fake_cred_agents.items()):
            actor = self.clients[cid]
            if state["activated"]:
                continue
            if step < state["dormant_until"] and state["n_leg_done"] < state["n_leg_target"]:
                if self.rng.uniform() < 0.1:
                    merchant = self.merchants[self.rng.choice(list(self.merchants.keys()))]
                    tx = self.fake_cred.execute_dormant_tx(step, actor, merchant, self.rng)
                    self.all_transactions.append(tx)
                    state["n_leg_done"] += 1
            elif step >= state["dormant_until"]:
                dest = self.clients[self.rng.choice(client_ids)]
                plafond = max(actor.mean_amount * 10, 50000)
                tx = self.fake_cred.execute_exfiltration(step, actor, dest, plafond, self.rng)
                self.all_transactions.append(tx)
                state["activated"] = True

        # --- Split Deposit ---
        if self.rng.uniform() < self.fraud_probas["split_deposit"] and self._split_dep_agents:
            agent = self._split_dep_agents[self.rng.choice(list(self._split_dep_agents.keys()))]
            client = self.clients[self.rng.choice(client_ids)]
            total_deposit = float(self.rng.uniform(2000, 80000))
            if agent.can_withdraw(total_deposit):
                txs = self.split_dep.execute(step, client, agent, total_deposit, self.rng)
                self.all_transactions.extend(txs)

        # --- Smurfing ---
        smurf_freq_mult = self.fraud_probas.get("smurfing_freq_mult", 1.0)
        for idx, net in enumerate(self.smurf_networks):
            if step >= self._smurf_next_op[idx]:
                total_X = float(self.rng.uniform(500000, 5000000))
                if net.emitter.balance >= total_X:
                    txs = net.run_laundering_operation(step, total_X, self.rng)
                    self.all_transactions.extend(txs)
                interval = (cfg["smurfing"]["operation_interval_days"] * 24) / max(smurf_freq_mult, 1e-3)
                self._smurf_next_op[idx] = step + int(self.rng.normal(interval, 24))

    # ------------------------------------------------------------------
    def run(self, n_steps: int = 720, verbose=True):
        for step in range(n_steps):
            for client in self.clients.values():
                self._run_client_legit_step(step, client)
            self._inject_fraud(step)
            if verbose and step % 10 == 0:
                print(f"step {step}/{n_steps} — {len(self.all_transactions)} tx cumulées", flush=True)
        return self.to_dataframe()

    # ------------------------------------------------------------------
    def to_dataframe(self) -> pd.DataFrame:
        rows = [{
            "step": t.step, "action": t.action, "amount": t.amount,
            "nameOrig": t.name_orig, "oldBalanceOrig": t.old_balance_orig,
            "newBalanceOrig": t.new_balance_orig, "nameDest": t.name_dest,
            "oldBalanceDest": t.old_balance_dest, "newBalanceDest": t.new_balance_dest,
            "isFraud": t.is_fraud, "isFlaggedFraud": t.is_flagged_fraud,
            "isUnauthorizedOverdraft": t.is_unauthorized_overdraft,
            "isSuccessful": t.is_successful, "fraudScenario": t.fraud_scenario,
        } for t in self.all_transactions]
        return pd.DataFrame(rows)

In [ ]:
df_raw = pd.read_csv("rawLog.csv")
engineer = FeatureEngineer(df_raw)
df_features = engineer.compute_all()
df_features.to_csv("featuresLog.csv", index=False)

print(df_features[["step", "action", "amount", "r1", "r2", "flag_anomalie",
                    "flag_nuit", "v1h", "delta_commission_ratio",
                    "is_mule_candidate", "var_agent_split", "rho_rupture",
                    "rho_refund", "rho_nouveau"]].tail(20))

In [ ]:
calib = SSEFraudCalibrator(
    param_dir="./paramFiles", fraud_config_path="./fraudScenariosConfig.yaml",
    seed=1000, n_clients=500, n_merchants=100, n_banks=10, n_mules=30,
    target_mid=0.23, n_steps=720, n_bins=30, n_seeds_per_eval=3)

result = calib.calibrate(maxiter=25)
print("\n=== Résultat calibration SSE ===")
print("probas :", result["probas"])
print("SSE final :", result["sse_final"])
print("convergé :", result["converged"])

with open("calibrated_probas.json", "w", encoding="utf-8") as f:
    json.dump(result["probas"], f, indent=2)

In [ ]:
sim = MoMTSimState(param_dir="./paramFiles", fraud_config_path="./fraudScenariosConfig.yaml",
                    seed=1000, n_clients=2000, n_merchants=300, n_banks=20, n_mules=60,
                    calibrated_probas_path="./calibrated_probas.yaml")
df = sim.run(n_steps=720)
df.to_csv("rawLog.csv", index=False)
print(df["isFraud"].mean(), "proportion de fraude générée")
print(df.loc[df["isFraud"], "fraudScenario"].value_counts(normalize=True))